In [ ]:
# ======================================================================
# SETUP — imports, paths, parameters
# ======================================================================
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats
from scipy.stats import linregress

from unito26.lob.lobster import LobsterFiles, TradingWindow
from unito26.lob.lobster_session import LobsterMarketSession
from unito26.lob.messages import GridDepth, TickGrid, ReportedDepth
from unito26.lob.statistics import SessionStatistics

DATA   = Path("../data")
TICKER = "AMZN"
DATE   = "2012-06-21"

MESSAGE_FILE   = DATA / f"{TICKER}_{DATE}_34200000_57600000_message_10.csv"
ORDERBOOK_FILE = DATA / f"{TICKER}_{DATE}_34200000_57600000_orderbook_10.csv"

WINDOW = TradingWindow(34200.0, 57600.0)
GRID   = TickGrid(0.01)
DEPTH  = ReportedDepth(10)

W_GRID = [1, 5, 30, 60, 300]
H_GRID = [1, 5, 30, 60, 300]
TAU    = 0.01

assert MESSAGE_FILE.exists(),   f"manca {MESSAGE_FILE}"
assert ORDERBOOK_FILE.exists(), f"manca {ORDERBOOK_FILE}"

In [ ]:
# ======================================================================
# Load LOBSTER files and build MarketSession
# ======================================================================
messages_df  = pd.read_csv(MESSAGE_FILE, header=None)
orderbook_df = pd.read_csv(ORDERBOOK_FILE, header=None)

assert messages_df.shape[1]  == 6
assert orderbook_df.shape[1] == 40
assert len(messages_df) == len(orderbook_df)

print(f"Messages: {len(messages_df):,}   Orderbook rows: {len(orderbook_df):,}")

lobster_files = LobsterFiles.parse(str(MESSAGE_FILE))
stats_spec = SessionStatistics((GridDepth(1), GridDepth(5)), (), ())
lobster_session = LobsterMarketSession.from_files(
    lobster_files, stats_spec, GRID, WINDOW
)
session = lobster_session.coarsened(True)

print("Session:", type(session).__name__)
print("stats columns:", list(session.stats.columns))

In [ ]:
# ======================================================================
# POINT i — manual fold + reconciliation vs LOBSTER orderbook
#   LOBSTER column order: AskPrice, AskSize, BidPrice, BidSize
#   orderbook_df.iloc[i] = state AFTER messages_df.iloc[i]
# ======================================================================
def fold_book(messages_df, orderbook_df, depth=10):
    first = orderbook_df.iloc[0].tolist()
    bids, asks = {}, {}

    for lv in range(depth):
        idx = lv * 4
        ap, as_ = int(first[idx]),     int(first[idx + 1])
        bp, bs  = int(first[idx + 2]), int(first[idx + 3])
        if ap > 0 and as_ > 0:
            asks[ap] = asks.get(ap, 0) + as_
        if bp > 0 and bs > 0:
            bids[bp] = bids.get(bp, 0) + bs

    orders = {}

    def snapshot():
        row = []
        top_bids = sorted(bids.items(), key=lambda kv: -kv[0])[:depth]
        top_asks = sorted(asks.items(), key=lambda kv:  kv[0])[:depth]
        for lv in range(depth):
            bp, bs  = top_bids[lv] if lv < len(top_bids) else (0, 0)
            ap, as_ = top_asks[lv] if lv < len(top_asks) else (0, 0)
            row.extend([ap, as_, bp, bs])
        return row

    def adjust(price, delta, side):
        book = bids if side == 1 else asks
        new = book.get(price, 0) + delta
        if new <= 0:
            book.pop(price, None)
        else:
            book[price] = new

    reconstructed = [snapshot()]

    for i in range(1, len(messages_df)):
        t, typ, oid, size, price, direction = map(int, messages_df.iloc[i].tolist())
        side = 1 if direction == 1 else -1

        if typ == 1:
            orders[oid] = (side, price, size)
            adjust(price, +size, side)
        elif typ in (2, 3, 4):
            if oid in orders:
                os_, op, osz = orders[oid]
                take = min(size, osz)
                adjust(op, -take, os_)
                if osz - take == 0:
                    del orders[oid]
                else:
                    orders[oid] = (os_, op, osz - take)
            else:
                adjust(price, -size, side)

        reconstructed.append(snapshot())

    return reconstructed


reconstructed = fold_book(messages_df, orderbook_df, depth=10)

n = len(reconstructed)
mismatches = 0
first_examples = []
for i in range(1, n):
    if reconstructed[i] != orderbook_df.iloc[i].tolist():
        mismatches += 1
        if len(first_examples) < 5:
            first_examples.append(i)

tot = n - 1
print(f"Compared {tot:,} states; mismatch = {mismatches:,} ({100*mismatches/tot:.4f}%)")
print("First mismatch indices:", first_examples)

In [ ]:
# Diagnostic: look at the first mismatch
i = first_examples[0]
print("message before:", messages_df.iloc[i-1].tolist())
print("message at    :", messages_df.iloc[i].tolist())
print("message after :", messages_df.iloc[i+1].tolist())
print()
print("reconstructed before :", reconstructed[i-1][:16])
print("LOBSTER before       :", orderbook_df.iloc[i-1].tolist()[:16])
print("reconstructed at     :", reconstructed[i][:16])
print("LOBSTER at           :", orderbook_df.iloc[i].tolist()[:16])

# Point i — Book reconstruction and reconciliation

I rebuild the order book using only the message file, then compare it
row by row with the orderbook file that LOBSTER gives us.

The fold starts from `orderbook_df.iloc[0]` (state after the first
message) and processes: submit (type 1) adds size; cancel / delete /
visible execution (types 2, 3, 4) remove size; hidden execution (type 5)
and halt (type 7) do nothing to the visible book.

**Result.** 269,641 mismatches out of 269,747 (99.9607%). The first 43
states match perfectly, then everything diverges from index 44 onward.
The first divergence occurs at index 44, after the first 43 reconstructed states 
match the LOBSTER book exactly. The event at the first mismatch is a type-4 visible execution, and the top four levels remain aligned, indicating that the initial discrepancy arises deeper in the book. A plausible explanation is the presence of orders already resting in the book at the beginning of the observation window, whose order-level history is not fully recoverable from messages observed after 09:30. Once such a discrepancy is introduced, subsequent updates can propagate it through the reconstructed state. Therefore, the reconciliation identifies the point and a plausible mechanism of divergence, rather than establishing a complete causal attribution for all subsequent mismatches. The session statistics used in Points ii–v come from LOBSTER's own aggregated book, so they are not affected by this mismatch.

In [ ]:
# ======================================================================
# POINT ii — 1s regular grid + OFI_{t,w} + I¹
# ======================================================================
stats_df = session.stats.copy()
stats_df.index = pd.to_timedelta(stats_df.index.to_numpy(dtype=float), unit="s")
stats_df = stats_df.sort_index()
stats_df = stats_df.rename(columns={"OrderFlowContribution": "e_n"})

g = stats_df.resample("1s").agg({
    "MidPrice":               "last",
    "Spread":                 "last",
    "QueueImbalance1":        "last",
    "QueueImbalance1Covered": "last",
    "TouchDepth":             "last",
    "e_n":                    lambda x: x.sum(min_count=1),
})
g["e_n"] = g["e_n"].fillna(0)

first_valid = g["MidPrice"].first_valid_index()
g = g.loc[first_valid:].copy()
state_cols = ["MidPrice", "Spread", "QueueImbalance1",
              "QueueImbalance1Covered", "TouchDepth"]
g[state_cols] = g[state_cols].ffill()

g["I1"] = g["QueueImbalance1"]

for w in W_GRID:
    g[f"OFI_{w}s"] = g["e_n"].rolling(window=w, min_periods=1).sum()

print(g[["MidPrice", "I1", "TouchDepth", "e_n"]
        + [f"OFI_{w}s" for w in W_GRID]].head(10))
print(f"\nRows: {len(g)}")

### Point ii — OFI and queue imbalance I¹

I put the LOBSTER statistics on a **regular 1-second grid**, so 6.5 hours
become 23,400 rows. If no message arrives in a second, `e_n = 0` (no
flow, not a missing value). For prices and sizes I use `ffill`: the book
doesn't change if no message arrives.

**OFI_{t,w}.** LOBSTER already gives me the signed contribution of each
event (`e_n`). So `OFI_{t,w}` is just the sum of `e_n` over the last w
seconds.

**I¹.** Already in `session.stats`: `(bid_size − ask_size) / (bid_size +
ask_size)`, between −1 and +1. Positive = more size on the bid.

In [ ]:
# ======================================================================
# POINT iii — ΔP_{t,w} = a + slope·OFI_{t,w} + ε
#             slope_theoric = τ/(2 S̄);  S̄_implied = τ / (2·slope)
#MidPrice is expressed here in tick units.
# ======================================================================
TAU_TICK = 1.0   # price unit is tick: MidPrice=22356.5, Spread=14.0 → 1 tick

bid1 = session.lobster_book["BidSize1"].mean()
ask1 = session.lobster_book["AskSize1"].mean()
td   = g["TouchDepth"].mean()
print(f"BidSize1 mean = {bid1:.1f}")
print(f"AskSize1 mean = {ask1:.1f}")
print(f"Sum           = {bid1 + ask1:.1f}")
print(f"TouchDepth mean = {td:.1f}")

S_bar_real = g["TouchDepth"].mean() / 2.0

rows = []
for w in W_GRID:
    dP = g["MidPrice"] - g["MidPrice"].shift(w)
    df = pd.concat([g[f"OFI_{w}s"].rename("OFI"), dP.rename("dP")],
                   axis=1).dropna()
    if len(df) < 100:
        continue
    slope, intercept, r_value, p_value, se = linregress(df["OFI"], df["dP"])
    S_implied = TAU_TICK / (2 * slope) if slope != 0 else np.nan
    rows.append(dict(
        w=w, n=len(df),
        slope=slope,
        slope_teorico=TAU_TICK / (2 * S_bar_real),
        R2=r_value**2, p_value=p_value,
        S_implied=S_implied, S_bar_real=S_bar_real,
        ratio=S_implied / S_bar_real if S_bar_real else np.nan,
    ))

contemp = pd.DataFrame(rows)
print("\n=== Point iii: contemporaneous ΔP vs OFI ===")
print(contemp.round(6).to_string(index=False))

### Point iii — Contemporaneous regression and implied depth

I regress `ΔP = P_t − P_{t−w}` on the cumulated OFI over the same window.
The theory says the slope should be `τ/(2S̄)`, so I invert it to get
the implied depth `S̄_implied`.

The slope is positive on every window, matching the theory. Its value
changes with w:

| w (s) | slope     | R²    | S̄_implied | S̄_real |
|-------|-----------|-------|-----------|--------|
| 1     | 0.001522  | 0.152 | 328       | 178    |
| 5     | 0.002063  | 0.236 | 242       | 178    |
| 30    | 0.003064  | 0.392 | 163       | 178    |
| 60    | 0.003040  | 0.408 | 164       | 178    |
| 300   | 0.002511  | 0.350 | 199       | 178    |

The real depth at the touch is around 178 shares per side. The implied
depth is close to that only for w = 30–60 s. At very short windows the
implied depth is bigger (240–330): the price moves **less** than the
theory predicts for a given OFI.

The reason is eq. (1.4.4): at short windows most of the flow just fills
or empties the queue that is already there. The price only moves when
the queue is fully consumed, so the residual ε captures the flow trapped
in the two touch queues. At 30-60 s, the estimated slope becomes much closer to 
the theoretical benchmark tau/(2 S-bar), suggesting that accumulated order flow 
over these horizons is more closely reflected in mid-price movements.
The R² values (0.15 to 0.41) are in line with the literature for
contemporaneous regressions on real data.

In [ ]:
# ======================================================================
# POINTS iv & v — R² of ΔP_{t+h} on OFI_{t,w} and on I¹
#   raw price (per assignment: P_{t+h} − P_t)
#   I¹ smoothed over w seconds, for symmetry with OFI_{t,w}
# ======================================================================
from scipy.stats import linregress

for w in W_GRID:
    g[f"I1_ma_{w}s"] = g["I1"].rolling(window=w, min_periods=1).mean()

def r2_surface(predictor_fn, W_GRID, H_GRID):
    rows = []
    for w in W_GRID:
        x = predictor_fn(w)
        for h in H_GRID:
            y = g["MidPrice"].shift(-h) - g["MidPrice"]   # raw price
            df = pd.concat([x.rename("x"), y.rename("y")], axis=1).dropna()
            if len(df) < 100:
                rows.append(dict(w=w, h=h, R2=np.nan, slope=np.nan, n=0))
                continue
            r = linregress(df["x"], df["y"])
            rows.append(dict(w=w, h=h, R2=r.rvalue**2, slope=r.slope, n=len(df)))
    return pd.DataFrame(rows)

ofi_grid = r2_surface(lambda w: g[f"OFI_{w}s"],   W_GRID, H_GRID)
i1_grid  = r2_surface(lambda w: g[f"I1_ma_{w}s"], W_GRID, H_GRID)

print("=== R² OFI ===")
print(ofi_grid.pivot(index="w", columns="h", values="R2").round(4))
print("\n=== R² I¹ ===")
print(i1_grid.pivot(index="w", columns="h", values="R2").round(4))
print("\n=== Slope OFI ===")
print(ofi_grid.pivot(index="w", columns="h", values="slope").round(6))
print("\n=== Slope I¹ ===")
print(i1_grid.pivot(index="w", columns="h", values="slope").round(6))

In [ ]:
for name, df in [("OFI", ofi_grid), ("I¹", i1_grid)]:
    px.imshow(
        df.pivot(index="w", columns="h", values="R2"),
        labels=dict(x="h (s)", y="w (s)", color="R²"),
        title=f"{name} forecasting — R² surface (raw price)",
        aspect="auto",
    ).show()

### Point iv — Forecasting OFI_{t,w} on the (w, h) grid

Now the other direction: does OFI today say something about the price
**in the future**? For each window w and horizon h, I regress
`P_{t+h} − P_t` on `OFI_{t,w}` and look at the R².

**Result.** Almost every cell has R² below 0.003. The only one that
stands out is `(w, h) = (300, 300)`, with **R² = 0.0195** and a
**negative slope** (−0.000624).

**What it means.** Predicting the future is much harder than explaining
the present: R² is 10–20 times smaller than in Point iii, and that's
normal. The one readable cell at (300, 300) has a negative sign, so it
is **mean reversion**: a positive OFI today predicts a slightly lower
price 5 minutes ahead, not a higher one. At short horizons (h<=5 s) the estimated relationship is negligible in this sample, with R-squared values close to zero.

### Point v — Forecasting with I¹ and comparison

Same exercise with I¹ (smoothed over w seconds to be comparable to
OFI_{t,w}). The maximum R² is **0.0090** at `(w, h) = (60, 300)`, about
half the OFI peak at (300, 300).

Side by side:

| (w, h)     | R²(OFI) | R²(I¹) | who wins |
|------------|---------|--------|----------|
| (1, 1)     | 0.0021  | 0.0020 | tie      |
| (60, 300)  | 0.0002  | 0.0090 | I¹       |
| (300, 300) | 0.0195  | 0.0077 | OFI      |

In this sample, the strongest OFI relationship occurs at (w,h)=(300,300) where the negative slope is consistent with mean reversion rather than momentum.

**Comparison with the simulated case.** One possible economic interpretation, 
consistent with the mechanisms discussed in the lecture notes, is that real 
traded books contain feedback mechanisms absent from the simulated setting, 
such as cancellations and liquidity adjustments in response to adverse order flow. 
This may weaken the persistence of queue imbalance relative to the idealised model.

So the ranking is **regime-dependent** (w, h), not a fixed hierarchy:
I¹ wins for medium w, OFI wins for long w, neither wins for short h.
This is what Remark 1.5.1 could not predict, because in the simulated
setting the hierarchy was imposed by the idealised book.

## Appendix — Robustness check: detrending the mid-price

A trailing 300-second moving average detrend was tested as a robustness
check. The result is **not** included in the main analysis (Points iv and v),
because a moving-average window introduces mechanical correlation between
predictor and dependent variable (overlapping-window artefact, not economic
signal). The main analysis stays on raw prices, as required by the assignment.

In [ ]:
# Robustness: remove the intraday trend — does the peak survive?
g["MidPrice_dt"] = g["MidPrice"] - g["MidPrice"].rolling(300, min_periods=60).mean()

def r2_surface_dt(predictor_fn, W_GRID, H_GRID):
    rows = []
    for w in W_GRID:
        x = predictor_fn(w)
        for h in H_GRID:
            y = g["MidPrice_dt"].shift(-h) - g["MidPrice_dt"]
            df = pd.concat([x.rename("x"), y.rename("y")], axis=1).dropna()
            if len(df) < 100:
                rows.append(dict(w=w, h=h, R2=np.nan, slope=np.nan, n=0))
                continue
            r = linregress(df["x"], df["y"])
            rows.append(dict(w=w, h=h, R2=r.rvalue**2, slope=r.slope, n=len(df)))
    return pd.DataFrame(rows)

print("=== R² OFI detrended ===")
print(r2_surface_dt(lambda w: g[f"OFI_{w}s"],   W_GRID, H_GRID).pivot(index="w", columns="h", values="R2").round(4))
print("\n=== R² I¹ detrended ===")
print(r2_surface_dt(lambda w: g[f"I1_ma_{w}s"], W_GRID, H_GRID).pivot(index="w", columns="h", values="R2").round(4))